Sascha Spors,
Professorship Signal Theory and Digital Signal Processing,
Institute of Communications Engineering (INT),
Faculty of Computer Science and Electrical Engineering (IEF),
University of Rostock,
Germany

# Data Driven Audio Signal Processing - A Tutorial with Computational Examples

Winter Semester 2025/26 (Master Course #24512)

- lecture: https://github.com/spatialaudio/data-driven-audio-signal-processing-lecture
- tutorial: https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise

Feel free to contact lecturer frank.schultz@uni-rostock.de

# Binary logistic regression model with one sigmoid layer
- we use **PyTorch** to train the model and to make predictions
- we use scikit-learn for data synthesis and split
- we use scikit-learn for statistical measures
- see [binary_logistic_regression_tf.ipynb](binary_logistic_regression_tf.ipynb) for a manual implementation and a TensorFlow implementation of the same problem

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sklearn
import torch

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.metrics import balanced_accuracy_score, accuracy_score

from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary

torch.__version__, sklearn.__version__  # last manual check with 2.8.0, 1.7.2

## Synthesis of Data

In [ ]:
# create some toy data
M = 100000  # number of samples per feature
N = 2  # number of features (excluding bias)
train_size = 0.8  # 80% of data are used for training, 20% for testing

X, Y = make_classification(
    n_samples=M,
    n_features=N,
    n_informative=N,
    n_redundant=0,
    n_classes=2,
    n_clusters_per_class=1,
    class_sep=1,
    flip_y=1e-2,
    random_state=8,
)
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, train_size=train_size, random_state=8
)
X_train, X_test

## Learning Parameters

In [ ]:
batch_size = X_train.shape[0]  # full batch
num_epochs = 50
learning_rate = 0.25

## Prepare Data for Torch

In [ ]:
Y_train = Y_train[:, np.newaxis]
Y_test = Y_test[: ,np.newaxis]

data_train = TensorDataset(torch.FloatTensor(X_train),
                           torch.FloatTensor(Y_train))
data_train_loader = DataLoader(dataset=data_train,
                               batch_size=batch_size,
                               shuffle=True)

## Define Torch Model

In [ ]:
class Model(torch.nn.Module):

    def __init__(self, input_size):
        super(Model, self).__init__()

        self.linear1 = torch.nn.Linear(input_size, 1)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x = self.linear1(x)
        x = self.sigmoid(x)
        return x
    
    def predict_class(self, x):
        x = torch.FloatTensor(x)
        pred = self.forward(x)
        return (pred >= 0.5).float()
    
model = Model(input_size=N)
summary(model, input_size=(batch_size, N))

In [ ]:
# init the model parameters (2 weights and 1 bias)
# to match with the results in
# binary_logistic_regression_tf.ipynb
# when we equivalently use
# rng = np.random.RandomState(1);  random_state=8;
# num_epochs = 500; learning_rate = 0.25; full batch
with torch.no_grad():
    model.linear1.weight[0, 0] = -0.165955990594852
    model.linear1.weight[0, 1] = 0.440648986884316
    model.linear1.bias[0] = -0.99977125036531
model.linear1.weight, model.linear1.bias

## Define Empirical Risk and Optimizer

In [ ]:
empirical_risk = torch.nn.BCELoss(reduction='mean') 
optimizer = torch.optim.SGD(model.parameters(),
                            lr=learning_rate)

## Train the Model

In [ ]:
for epoch in range(num_epochs):
    if (epoch+1) % 10 == 0:
        print('epoch:', epoch+1)
    for i, batch in enumerate(data_train_loader, 1):
        X, Y = batch[0], batch[1]
        Y_pred = model.forward(X)
        loss = empirical_risk(Y_pred, Y)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

## Check the Model Parameters

In [ ]:
model.linear1.weight, model.linear1.bias

## Test the Model

### Empirical Risk

In [ ]:
with torch.no_grad():
    er = empirical_risk(model.forward(
        torch.FloatTensor(X_test)),
        torch.FloatTensor(Y_test))
    print(er)
    er = empirical_risk(model.forward(
        torch.FloatTensor(X_train)),
        torch.FloatTensor(Y_train))
    print(er)

### Confusion Matrix

In [ ]:
with torch.no_grad():
    print(confusion_matrix(
        y_true=Y_train,
        y_pred=model.predict_class(X_train),
        normalize=None))
    print(confusion_matrix(
        y_true=Y_train,
        y_pred=model.predict_class(X_train),
        normalize='all')*100)
    print()    
    print(confusion_matrix(
        y_true=Y_test,
        y_pred=model.predict_class(X_test),
        normalize=None))
    print(confusion_matrix(
        y_true=Y_test,
        y_pred=model.predict_class(X_test),
        normalize='all')*100)

### Precision, Recall, F1Score, Support

In [ ]:
with torch.no_grad():
    p, r, f, s = precision_recall_fscore_support(y_true=Y_train,
                                                 y_pred=model.predict_class(X_train))
    print(p, r, f, s)
    p, r, f, s = precision_recall_fscore_support(y_true=Y_test,
                                                 y_pred=model.predict_class(X_test))
    print(p, r, f, s)

### Accuracy, Balanced Accuracy

We have a very balanced data set, hence both are very close

In [ ]:
with torch.no_grad():
    a = accuracy_score(y_true=Y_train,
                       y_pred=model.predict_class(X_train))
    ba = balanced_accuracy_score(y_true=Y_train,
                                 y_pred=model.predict_class(X_train))
    print(a, ba)

    a = accuracy_score(y_true=Y_test,
                                  y_pred=model.predict_class(X_test))
    ba = balanced_accuracy_score(y_true=Y_test,
                                 y_pred=model.predict_class(X_test))
    print(a, ba)

## Plot Data Points and Decision Plane

In [ ]:
def my_sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

In [ ]:
# get model parameters
w = model.linear1.weight.detach().numpy()
b = model.linear1.bias.detach().numpy()

# plot
levels = [0.0, 0.05, 0.45, 0.5, 0.55, 0.95, 1]
if N == 2:  # 2D plot of data and classification line when having two features
    f1, f2 = np.arange(-6, 6, 0.1), np.arange(-6, 6, 0.1)
    xv, yv = np.meshgrid(f1, f2)
    tmp = my_sigmoid(w[0, 0] * xv + w[0, 1] * yv + b[0])
    # hard decision boundary:
    # tmp[tmp < 0.5], tmp[tmp >= 0.5] = 0, 1

    plt.figure(figsize=(10, 10))
    plt.subplot(2, 2, 1)
    plt.plot(X_train[Y_train[:, 0] == 1, 0],
             X_train[Y_train[:, 0] == 1, 1],
             "o", color='orangered', ms=1)
    plt.contourf(f1, f2, tmp, levels=levels, cmap="RdBu_r")
    plt.axis("equal")
    plt.colorbar()
    plt.title("training " + str(X_train.shape))
    plt.xlabel("feature 1")
    plt.ylabel("feature 2")

    plt.subplot(2, 2, 2)
    plt.plot(X_train[Y_train[:, 0] == 0, 0],
             X_train[Y_train[:, 0] == 0, 1],
             "o", color='dodgerblue', ms=1)
    plt.contourf(f1, f2, tmp, levels=levels, cmap="RdBu_r")
    plt.axis("equal")
    plt.colorbar()
    plt.title("training " + str(X_train.shape))
    plt.xlabel("feature 1")
    plt.ylabel("feature 2")

    plt.subplot(2, 2, 3)
    plt.plot(X_test[Y_test[:, 0] == 1, 0],
             X_test[Y_test[:, 0] == 1, 1],
             "o", color='orangered', ms=1)
    plt.contourf(f1, f2, tmp, levels=levels, cmap="RdBu_r")
    plt.axis("equal")
    plt.colorbar()
    plt.title("test " + str(X_test.shape))
    plt.xlabel("feature 1")
    plt.ylabel("feature 2")

    plt.subplot(2, 2, 4)
    plt.plot(X_test[Y_test[:, 0] == 0, 0],
             X_test[Y_test[:, 0] == 0, 1],
             "o", color='dodgerblue', ms=1)
    plt.contourf(f1, f2, tmp, levels=levels, cmap="RdBu_r")
    plt.axis("equal")
    plt.colorbar()
    plt.title("test " + str(X_test.shape))
    plt.xlabel("feature 1")
    plt.ylabel("feature 2")

## Copyright

- the notebooks are provided as [Open Educational Resources](https://en.wikipedia.org/wiki/Open_educational_resources)
- feel free to use the notebooks for your own purposes
- the text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/)
- the code of the IPython examples is licensed under the [MIT license](https://opensource.org/licenses/MIT)
- please attribute the work as follows: *Frank Schultz, Data Driven Audio Signal Processing - A Tutorial Featuring Computational Examples, University of Rostock* ideally with relevant file(s), github URL https://github.com/spatialaudio/data-driven-audio-signal-processing-exercise, commit number and/or version tag, year.